# The Term Structure Of Interest Rates Drives Yields To Maturity 📓 📈 🎯

<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Four-Chapter-Three/blob/main/colab/Colab_Calculating_Yields_To_Maturity.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Four-Chapter-Three/main?urlpath=lab/tree/notebooks/Calculating_Yields_To_Maturity.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>
  
  <a href="https://patrickjhess.github.io/Volume-Four-Chapter-Three/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>

This notebook demonstrates the relation between the term structure of interest rates and yield to maturity. Data is downloaded from FEDInvest data and yields to maturity are calculated for eight bonds with maturities between one and thirty years. Six different term structures regimes are used to price the bonds. The base case is the estimate of Chapter Two for May 15, 2026. The other five regimes vary the level, slope, and shape of the term structure. The regimes range from increases in the rate and slope to a negative slope and increase in the short rate.

In order to find the bonds' yields to maturity, we use the `optimize.newton` function of SciPy. The function searches for the solution by calculating the error of the function and adjusting guesses of the yield to maturity:
<br>

$$\text{Error}=\sum_{t=1}^{T}\text{Cash Flow}{t}\times e^{-\text{guess}_i\times t}-\text{Bond Price}$$



$$\displaystyle guess_{i+1} = \text{guess}_i\times-\frac{\text{Error}}{\sum_{t=1}^{T}-t\times\text{Cash Flow}{t}\times e^{-\text{guess}_i\times t}}$$
<br>

The function continues to search until the error is less than one one-thousandth of a basis point.

The yields to maturity of each are calculated for the six term structures and the results are plotted along with the term structures that generated the yields to maturity. As expected, the yields to maturity maintain the general shape of their matching term structures, but lag behind the term structures.

:::{important} [ ▼ ] How to use this page: Run, Copy, & Download
:class: dropdown

<ul>
  <li><b>⏻ Run code right here:</b> Click the <b>Power Button</b> icon at the top of the screen to activate <b>Live Code</b>.</li>
  <li><b>📋 Copy code:</b> Hover over any code block and click the <b>Clipboard icon</b> in the top-right corner.</li>
  <li><b>📥 Download this file:</b> Click the <b>Download icon</b> (downward arrow) at the top right of the screen to save this exact notebook to your computer.</li>
</ul>
:::

<details>
<summary><b>🤔 Notebook Setup: Why the "Try/Except" Imports?</b></summary>

**The Goal:**
To ensure this notebook runs perfectly whether you are using **Google Colab**, a local **Jupyter instance**, or a remote server without you having to manually install software.

**Key Concepts in this Section:**
*   **Standard Libraries:** Modules like `os`, `sys`, and `datetime` come "in the box" with Python. We use them for system tasks and date math.
*   **External Libraries:** NumPy and Pandas are the "heavy hitters" for data. They aren't always installed by default.
*   **The `try/except` Logic:** This is a safety net.
    1. We **try** to import the library.
    2. If it fails (because it's not installed), the **except** block triggers a `%pip install` to download it automatically.
*   **Aliasing (`as np`):** We rename `numpy` to `np` to save keystrokes. In professional finance code, `np` and `pd` are the universal shorthand.

</details>

## 🛠️ Preparing the Notebook
<details>
<summary><b>👉 Click to Expand: 📦 Importing Libraries, Modules, and Functions</b></summary>


As a best practice, we always begin by importing our necessary dependencies in the very first code cell. Here we import the `datetime` module, the `pandas, numpy,scipy`, and `altair` libraries ✨

```
from datetime import datetime, date


try:
    import numpy as np
except:
    %pip -q install numpy
    import numpy as np

try:
    import pandas as pd
except:
    %pip -q install pandas
    import pandas as pd

# Import the scipy library to find roots of present value function
try:
  import scipy.optimize as optimize
except:
  %pip install -q scipy
  import scipy.optimize as optimize

# Import Altair library to visualize spot rates and yields to maturity
try:
  import altair as alt
except:
  %pip install -q altair
  import altair as alt
```
SciPy stands for scientific Python and is a powerful library built around NumPy,  In this notebok, the function `minimze`  (a subset of SciPy) is used to find the estimates of yield to maturity.  Altair is Python library used for data visualization.

**👀 Keep an eye out**: As we progress, pay attention to how the `financial_quant` package is imported as `fq`, and how every reference to its functions begins with fq.. 💡 This follows the exact same standard practice we demonstrated in Chapter One with NumPy (np) and Pandas (pd).

</details>

In [13]:

from datetime import datetime, date


try:
    import numpy as np
except:
    %pip -q install numpy
    import numpy as np

try:
    import pandas as pd
except:
    %pip -q install pandas
    import pandas as pd

# Import the scipy library to find roots of present value function
try:
  import scipy.optimize as optimize
except:
  %pip install -q scipy
  import scipy.optimize as optimize

# Import Altair library to visualize spot rates and yields to maturity
try:
  import altair as alt
except:
  %pip install -q altair
  import altair as alt

## 📦 Getting Functions from financial_quant package

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;">🔌 Professional Packaging: How GitHub Installations Work</b></summary>
<br>
<p><b>The Logic:</b><br>
Usually, Python looks for modules as <code>.py</code> files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.</p>

<p><b>The Workflow:</b></p>
<ol>
<li><b>Fetch & Build:</b> The <code>%pip install git+https://...</code> command tells your Jupyter environment to clone the repository from GitHub and install the <code>financial_quant</code> package directly into your system's site-packages directory.</li>
<li><b>Import:</b> <code>import financial_quant as fq</code> loads the package into your notebook's memory and assigns it the quick alias <code>fq</code>.</li>
<li><b>Routing:</b> Behind the scenes, a special file called <code>__init__.py</code> acts as the package's "front door." It automatically gathers complex tools from deeply nested folders (like our fixed-income models and chart visualizers) and serves them up directly to the surface.</li>
<li><b>Execute:</b> You don't have to worry about where the files live. You just type <code>fq.one_y_axis() or fq.calc_ytm()</code>, and Python immediately knows where to route the request.</li>
</ol>

<p><b>Why do this?</b><br>
This is exactly how professional software engineering teams manage and distribute code. It keeps your notebooks incredibly clean, ensures everyone is using the exact same version of the math models, and guarantees your code is 100% portable to any cloud environment</p>
</details>

In [14]:
%pip install -q git+https://github.com/PatrickJHess/quant_repo.git
import financial_quant as fq

Note: you may need to restart the kernel to use updated packages.


## 🛠️📋 Steps in our analysis

*  1. **Get bond data** from FEDInvest that is used to demonstrate how yield to maturity relates to the term structure.
* 2. **Value the bonds** downloded from FEDInvest for alternative versions of the term structure.
  * Create the DataFrame `df_payoff` with columns holding the payment amount for each bond on the unique payment dates of all the bonds. Most dates for a bond will be zero.
  * Create the DataFrame `df_zeros` with columns holding the zero prices (discount factors) for each unique payment date under the different term structure regimes..
  * Create the DataFrame `bond_value` as the product of `df_zeros` and `df_payoff`. (Note: This is effectively a dot product representing the present value of future cash flows).
* 3. **Calculate Yields**: Create the DataFrame `df_ytm` with yields to maturity of the bonds for each term structure regime. The calculations use the SciPy function `optimize.newton`.

* 4. **Visualize**: The library Altair is used to produce a side-by-side plot of each term structure regime and the resulting yields to maturity.

## 📊 1. The Bond Data From FEDInvest

Eight bonds with maturities closely matching one, two, three, five, seven, ten, twenty, and thirty years are downloaded from FEDInvest. They have a settlement date of May 15, 2026, which aligns exactly with the date of the Nelson-Siegel baseline term structure estimates.

### 1.a 📥🧹 Getting and standardizing the data from FEDInvest

In [3]:
# bond data date
my_date=datetime(2026,5,14)
# fetch Treasury data for a specific date
df, stamp,settlement = fq.FEDInvest(my_date)

#display price and date and settlement
display(f'Date of prices {stamp}  Settlement date  {settlement}')

# standardize the data
clean_df=fq.clean_FEDInvest(df)
display(clean_df)

'Date of prices 2026-05-14  Settlement date  2026-05-15'

,SECURITY TYPE,Coupon,Price Ask,Price Bid,END OF DAY
MATURITY DATE,,,,,
2026-05-19,MARKET BASED BILL,0.000,0.000000,99.950694,99.960556
2026-05-21,MARKET BASED BILL,0.000,99.931167,99.930972,99.940667
2026-05-26,MARKET BASED BILL,0.000,99.881667,99.881333,99.891528
2026-05-28,MARKET BASED BILL,0.000,99.861944,99.861556,99.871444
2026-05-31,MARKET BASED NOTE,4.875,0.000000,100.031250,100.031250
...,...,...,...,...,...
2055-02-15,MARKET BASED BOND,4.625,93.984375,93.968750,93.750000
2055-05-15,MARKET BASED BOND,4.750,95.921875,95.906250,95.656250
2055-08-15,MARKET BASED BOND,4.750,95.984375,95.968750,95.718750


#### 1.b 🗓️📍 Set the target maturity dates
* **Settlement Date**: Initializes the baseline settlement date as May 15, 2026.

* **Time Horizons**: It creates a NumPy array of pandas DateOffset objects corresponding to standard benchmark yield curve maturities: 1, 2, 3, 5, 7, 10, 20, and 30 years.

* **Date Calculation**: Converts the settlement date to a pandas datetime object and adds the array of offsets. This broadcasts the addition across the array, calculating the exact future calendar dates for all eight maturities simultaneously.

* **DatetimeIndex**:The resulting list of dates are converted to a pandas DatetimeIndex.

In [4]:
# assumed settlement date
settlement=date(2026,5,15)

# newly issued constant coupon maturity calculated from settlement
# SOFR overnight maturity, FRED maturity dates as reported
maturity_dates =pd.DatetimeIndex(pd.to_datetime(settlement) + np.array([
    pd.DateOffset(years=1),  pd.DateOffset(years=2),
    pd.DateOffset(years=3),  pd.DateOffset(years=5),  pd.DateOffset(years=7),
    pd.DateOffset(years=10), pd.DateOffset(years=20), pd.DateOffset(years=30)
]))


### 1.c 🗓️ Selecting bonds that mature on or about the dates of `maturity_dates`

* The greatest maturity date of `clean_df` may be less than the final target maturity of thirty years. If so, we want to make sure that the longest maturity bond available is still included.

* The first calculation of `possible dates` will exclude the target maturity dates that exceed the maximum maturity in our dataset (as is the case here with the thirty-year mark). The `union` method in the second calculation adds the last available maturity if it was excluded in the first step.

* The `searchsorted` method determines the integer location of an ascending pandas `DatetimeIndex`.

**Note**: The last bond included in `bond_data` has a maturity of twenty-nine years and nine months, which is less than the thirty-year mark of the last value in `maturity_dates`. That specific bond is added back into our selection using the `union` method.

In [5]:
# possible dates will exclude the last date it it's greater than last available
possible_dates = maturity_dates[maturity_dates <= clean_df.index[-1]]

# datetime index union works just like a set (all values unique)
possible_dates = possible_dates.union(pd.Index([clean_df.index[-1]]))

# searchsorted defaults to the first integer location of the value
matching_rows = clean_df.index.searchsorted(possible_dates)

# matching rows first location of match and last value if maturity then the lst
bond_data = clean_df.iloc[matching_rows]

display(bond_data)

,SECURITY TYPE,Coupon,Price Ask,Price Bid,END OF DAY
MATURITY DATE,,,,,
2027-05-15,MARKET BASED NOTE,2.375,98.578125,98.56250,98.53125
2028-05-15,MARKET BASED NOTE,2.875,97.890625,97.87500,97.84375
2029-05-15,MARKET BASED NOTE,2.375,95.390625,95.37500,95.34375
2031-05-15,MARKET BASED NOTE,1.625,88.781250,88.75000,88.68750
2033-05-15,MARKET BASED NOTE,3.375,94.578125,94.56250,94.46875
2037-02-15,MARKET BASED BOND,4.750,102.640625,102.56250,102.43750
2046-05-15,MARKET BASED BOND,2.500,67.937500,67.90625,67.75000
2056-02-15,MARKET BASED BOND,4.750,96.046875,96.03125,95.78125


## ⚖️ 2. Pricing the bonds with the term structure

### 2.a 🧮 Creating the DataFrame df_payoff
* **Helper Function** `create_payoff_df`: As used in Chapter Two of this Volume and Chapter Four of the Volume Term Structure of Interest Rates, this function calculates a DataFrame with a row for each bond's payments on every unique payment date across all the bonds.

* **Transposing the DataFrame**: Transposing the DataFrame created by `create_payoff_df` results in a new DataFrame with columns representing the bonds' payment amounts and an index of the unique payment dates.

In [6]:
# generate the initial payoff matrix using the custom helper function
df_payoff = fq.create_payoff_df(bond_data, settlement=settlement)

# set the row index to match the original bond identifiers from bond_data
df_payoff.index = bond_data.index

# transpose the DataFrame so that unique payment dates become the row index,
# and the individual bonds become the columns
df_payoff = df_payoff.T

# name the newly created datetime index for clarity in the visual output
df_payoff.index.name = 'Payment Dates'

# render the formatted DataFrame in the Jupyter Notebook
display(df_payoff)

MATURITY DATE,2027-05-15,2028-05-15,2029-05-15,2031-05-15,2033-05-15,2037-02-15,2046-05-15,2056-02-15
Payment Dates,,,,,,,,
2026-08-17,0.0000,0.0000,0.0000,0.0000,0.0000,2.375,0.00,2.375
2026-11-16,1.1875,1.4375,1.1875,0.8125,1.6875,0.000,1.25,0.000
2027-02-16,0.0000,0.0000,0.0000,0.0000,0.0000,2.375,0.00,2.375
2027-05-17,101.1875,1.4375,1.1875,0.8125,1.6875,0.000,1.25,0.000
2027-08-16,0.0000,0.0000,0.0000,0.0000,0.0000,2.375,0.00,2.375
...,...,...,...,...,...,...,...,...
2054-02-17,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00,2.375
2054-08-17,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00,2.375
2055-02-16,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00,2.375


### 2.b ⚙️ Create The DataFrame Of Zero Prices

#### 2.b.1 💹 The term structure regimes
* **The Base Case**:The starting parameter assume an upward-sloping curve with small intermediate hump.

  * *Parameters*: $\beta_0 = 5.68\%$, $\beta_1 = -2.08\%$, $\beta_2 = -0.014$, $\tau = 3.0896$
  * *Resulting Rates*: Short Rate = 3.60% | Long Rate = 5.68% | Slope = +2.08%
* **Increase All Rates By 100 Basis Points (Parallel)**: Constant Slope (Parallel Shift)
 bump the entire level of the curve up by 100 basis points the distance
 between the short and long rates is unchanged maintaining a constant slope.

  * *Parameters*: $\beta\_0 = 6.68\%$, $\beta_1 = -2.08$\%, $\beta\_2 = -0.014$, $\tau = 3.0896$  
  * *Resulting Rates*: Short Rate = 4.60%, Long Rate = 6.68% , Slope = 2.08%  

* **Increase Rates, Increasing Slope (Bear Steepner)**:Short rates rise dramatically due to Fed hikes to SOFR but with a muted effect on long rates.
  * *Parameters*: $\beta_0 = 7.5\%$, $\beta_1 = -3.00\%$, $\beta_2 = -0.014$, $\tau = 3.0896$  
  * *Resulting Rates*: Short Rate = 4.50%, Long Rate= 7.50%, Slope = +3.000%  


* **Increase Short Rates, Decreasing Slope (Bear Flattener)**:Short rates rise dramatically due to Fed hikes to SOFR but with a muted effect on long rates.
  * *Parameters*: $\beta_0 = 5.80\%$, $\beta_1 = -0.50\%$, $\beta_2 = -0.014$, $\tau = 3.0896$  
  * *Resulting Rates*: Short Rate = 5.30%, Long Rate= 5.80%, Slope = +0.50%  

* **Negative Slope (Inverted) with a Belly Dip**:
Decrease dramatic cut in the long-term rate ($\beta_0$), make $\beta_1$ positive inverting the curve, and cause a negative curvature ($\beta_2$) resulting in a "dip" in the intermediate years (often viewed to be a recession indicator).
  * *Parameters:* $\beta_0 = 3.50\%$, $\beta_1 = 1.50\%$, $\beta_2 = -0.50$, $\tau = 3.0896$  
  * *Resulting Rates:* Short Rate \= 5.00% | Long Rate \= 3.50% | Slope \= \-1.50%

* **Flattening the Curve using $\tau$ (Tau)**:Instead of changing the starting or ending rates, increasing $\tau$ stretches the curve out. It takes longer for spot rates to climb to the long rate.

  * *Parameters*: $\beta_0 = 5.68\%$, $\beta_1 = -2.08\%$, $\beta_2 = -0.014$, $\tau = 4.0$
  * *Resulting Rates*: Short Rate = 3.60% | Long Rate = 5.68% | Slope = +2.08%



####  2.b.2 🐼 (Panda) Creating the DataFrame `df_zeros`
* **Calculating Time to Maturity**: First, we calculate the exact time to maturity (in years) for every unique payment date in our dataset by finding the difference in days from the settlement date (May 15, 2026) and dividing by 365.25.
* **Calculating Discount Factors**: For each Nelson-Siegel term structure scenario, we compute the corresponding spot rates for these specific maturities. We then convert these rates into zero-coupon prices (discount factors) using continuous compounding ($e^{-r \times t}$).
* **Structuring the Output**: The resulting discount factors are appended as rows into the df_zeros DataFrame, where each row represents a distinct scenario and each column corresponds to a unique payment date.

In [8]:
# define alternative term structure scenarios using Nelson-Siegel parameters
# each tuple represents (beta_0, beta_1, beta_2, tau)
scenarios = [
    (0.0568, -0.0208, -0.014, 3.0896), # Base Case
    (0.0668, -0.0208, -0.014, 3.0896), # Parallel Increase
    (0.0750, -0.0300, -0.014, 3.0896), # Bear Steepner
    (0.0580, -0.0050, -0.014, 3.0896), # Bear Flattener
    (0.0350,  0.0150, -0.025, 3.0896), # Inverted
    (0.0568, -0.0208, -0.014, 8.0000)  # Flatten Curve
]

# descriptive names corresponding to each parameter tuple above
scenario_names = [
    'Base Case',
    'Parallel Increase',
    'Bear Steepner',
    'Bear Flattener',
    'Inverted',
    'Flatten Curve'
]

# initialize an empty DataFrame with columns matching the unique payment dates
df_zeros = pd.DataFrame(columns=df_payoff.index)

# calculate the time to maturity in years for each payment date (using May 15, 2026 as settlement)
mat_years = (pd.to_datetime(df_payoff.index) - pd.to_datetime('2026-05-15')).days / 365.25

# iterate through each term structure scenario
for scenario in scenarios:
    # calculate spot rates for the exact payment maturities
    rates, years = fq.ns_spot_rates(scenario, mat_years)

    # calculate the discount factor using continuous compounding and add it as a new row
    df_zeros.loc[len(df_zeros)] = np.exp(-rates * years)

# apply the descriptive scenario names to the index
df_zeros.index = scenario_names
df_zeros.index.name = 'Scenario'

# Render the formatted DataFrame
display(df_zeros)

Payment Dates,2026-08-17,2026-11-16,2027-02-16,2027-05-17,2027-08-16,2027-11-15,2028-02-15,2028-05-15,2028-08-15,2028-11-15,...,2051-08-15,2052-02-15,2052-08-15,2053-02-18,2053-08-15,2054-02-17,2054-08-17,2055-02-16,2055-08-16,2056-02-15
Scenario,,,,,,,,,,,,,,,,,,,,,
Base Case,0.990704,0.981640,0.972407,0.963307,0.954035,0.944695,0.935185,0.925821,0.916192,0.906510,...,0.265304,0.257825,0.250634,0.243453,0.236809,0.230060,0.223677,0.217403,0.211370,0.205441
Parallel Increase,0.988157,0.976680,0.965061,0.953676,0.942147,0.930602,0.918916,0.907477,0.895779,0.884083,...,0.206100,0.199284,0.192763,0.186284,0.180319,0.174290,0.168617,0.163068,0.157759,0.152568
Bear Steepner,0.988317,0.976822,0.965030,0.953344,0.941393,0.929322,0.917014,0.904886,0.892417,0.879891,...,0.172383,0.165995,0.159908,0.153887,0.148365,0.142807,0.137598,0.132525,0.127691,0.122982
Bear Flattener,0.986541,0.973829,0.961255,0.949194,0.937213,0.925425,0.913684,0.902352,0.890914,0.879610,...,0.245126,0.238071,0.231292,0.224528,0.218272,0.211922,0.205919,0.200022,0.194356,0.188791
Inverted,0.987619,0.976477,0.965942,0.956250,0.946985,0.938186,0.929698,0.921737,0.913901,0.906328,...,0.426099,0.418663,0.411433,0.404134,0.397306,0.390293,0.383586,0.376923,0.370445,0.364008
Flatten Curve,0.990750,0.981822,0.972822,0.964043,0.955192,0.946365,0.937467,0.928786,0.919938,0.911116,...,0.306427,0.298189,0.290239,0.282273,0.274879,0.267343,0.260194,0.253148,0.246354,0.239660


### 💵 2.c The Final Result: Theoretical Bond Values

* **The Matrix Multiplication (@)**: By using the @ operator, we take the dot product of the discount factors (df_zeros) and the cash flows (df_payoff). This mathematical operation efficiently calculates the present value of all future cash flows for every bond, under every scenario, in a single step.

* **Structuring the Output**: The resulting DataFrame, bond_value, contains the theoretical prices of our eight bonds across the six different term structure regimes.


In [9]:
# calculate present value using matrix multiplication (Discount Factors @ Cash Flows)
bond_value = df_zeros @ df_payoff

# apply the descriptive scenario names to the index and display
bond_value.index = scenario_names
display(bond_value)

MATURITY DATE,2027-05-15,2028-05-15,2029-05-15,2031-05-15,2033-05-15,2037-02-15,2046-05-15,2056-02-15
Base Case,98.640280,98.066875,95.394343,88.208361,93.489380,101.319313,67.577765,94.751528
Parallel Increase,97.659878,96.164780,92.657661,84.071330,87.829540,93.194026,58.537368,81.855703
Bear Steepner,97.626484,95.899890,92.044967,82.635632,85.458803,89.214659,53.561049,74.935306
Bear Flattener,97.202996,95.626991,92.289906,84.428178,89.085077,96.205137,63.536239,89.223297
Inverted,97.920119,97.625643,95.844731,91.280273,99.699410,113.581919,87.244668,126.019346
Flatten Curve,98.715061,98.371360,96.064335,89.862041,96.385643,106.480897,73.997317,103.185871


## 🧮 3. Calculating Yield to Maturity (YTM)

This section uses the custom module function `calc_ytm` to calculate the Yield to Maturity (YTM) using the Newton-Raphson method (`scipy.optimize.newton`). By setting up an objective function and its mathematical derivative, the algorithm iteratively hones in on the exact discount rate that equates the present value of the bond's cash flows to its market price.
* **A quick note on the math**:
  * **Objective Function**: Represents the pricing error. We want to find the yield ($y$) that makes this difference zero:
  <br>

  $$\text{Error} = \left( \sum_{i} CF_i \times e^{-y \times t_i} \right) - \text{Price}$$
  <br>

  * **Derivative**: The first derivative of the present value with respect to the yield ($y$), which helps the optimizer take smart, efficient steps toward the correct answer:
  <br>

  $$\frac{d(PV)}{dy} = \sum_{i} -t_i \times CF_i \times e^{-y \times t_i}$$

:::{dropdown} 🔍 Click to see `calc_ytm`
:icon: search

```python
def calc_ytm(guesses, cash_flows, mat_years, prices):
    """
    Calculates the Yield to Maturity (YTM) using the Newton-Raphson method.
    """
    # use np.isscalar to check if the user passed a single number instead of an array
    if np.isscalar(guesses):
        # If a single guess is provided, broadcast it to match the length of the prices array
        guesses = [guesses] * len(prices)

    def ytm_objective(y, cash_flows, times, market_price):
        """Objective function: difference between modeled PV and actual price."""
        # Continuous compounding: PV = CF * e^(-y * t)
        pv = sum([cf * np.exp(-y * t) for cf, t in zip(cash_flows, times)])
        return pv - market_price

    def ytm_derivative(y, cash_flows, times, market_price):
        """Derivative of the objective function with respect to the yield (y)."""
        # Power rule applied to e^(-y * t) brings down the (-t)
        return sum([-t * cf * np.exp(-y * t) for cf, t in zip(cash_flows, times)])
    
    # Run the Newton-Raphson optimization
    ytm = optimize.newton(
        func=ytm_objective,
        x0=guesses,
        fprime=ytm_derivative,
        args=(cash_flows, mat_years, prices)
    )
    
    return ytm
```
:::

In [11]:
df_ytm=pd.DataFrame(columns=df_payoff.columns)
# iterate through each scenario

for name in df_payoff.columns:
    filtered = df_payoff[name][df_payoff[name] != 0]

    # 1. Extract raw arrays using .to_numpy()
    # (If you skip this, Pandas will break the 2D math inside ytm_objective)
    maturity = (pd.DatetimeIndex(filtered.index) - pd.to_datetime('2026-05-15')).days.to_numpy() / 365.25
    cash_flows = filtered.to_numpy()
    prices = bond_value[name].to_numpy()
    df_ytm[name]=fq.calc_ytm(0.05,cash_flows,maturity,prices)

df_ytm.index=scenario_names
df_ytm.columns=[col.strftime('%Y-%m-%d') for col in df_ytm.columns]
display(df_ytm.style.format('{:.3%}'))

,2027-05-15,2028-05-15,2029-05-15,2031-05-15,2033-05-15,2037-02-15,2046-05-15,2056-02-15
Base Case,3.720%,3.849%,3.979%,4.224%,4.417%,4.673%,5.042%,5.108%
Parallel Increase,4.720%,4.849%,4.979%,5.224%,5.416%,5.669%,6.031%,6.081%
Bear Steepner,4.754%,4.990%,5.207%,5.583%,5.855%,6.194%,6.660%,6.705%
Bear Flattener,5.190%,5.135%,5.116%,5.136%,5.188%,5.288%,5.462%,5.500%
Inverted,4.454%,4.079%,3.817%,3.513%,3.393%,3.334%,3.354%,3.384%
Flatten Curve,3.644%,3.690%,3.739%,3.838%,3.930%,4.088%,4.432%,4.571%


## 🎨 4. Visualizing Spot Rates and Yields To Maturity

Here we use the modern library Altair to present spot rates and yields to maturity as side-by-side graphs. While Matplotlib is the traditional powerhouse of Python plotting, Altair is a declarative library based on the "Grammar of Graphics." With Altair, we can easily create side-by-side visualizations that share the exact same y-axis scale, making cross-chart comparisons much easier to comprehend.

In [12]:
import altair as alt

# create DataFrame of spot rates (df_spot_rates) from DataFrame of zeros (df_zeros)
df_spot_rates=np.log(1/df_zeros)/mat_years

# prepare left data (first plot) (Spot Rates)
# make the transpose of df_spot rates left plot...rename index
df_left = df_spot_rates.T.reset_index(names='Date')

# 'date' objects to Pandas datetime objects
df_left['Date'] = pd.to_datetime(df_left['Date'])

# MELT IN PANDAS (Turns wide data into Altair-ready long data)
# maps the various scenarios into a single 'Scenario' column and rates into a 'Rate' column
df_left_long = df_left.melt(id_vars='Date', value_vars=scenario_names, var_name='Scenario', value_name='Rate')

# prepare Right Data (Yield To Maturity) ---
df_right = df_ytm.T.reset_index(names='Date')

# 'date' objects to Pandas datetime objects
df_right['Date'] = pd.to_datetime(df_right['Date'])

# MELT IN PANDAS
df_right_long = df_right.melt(id_vars='Date', value_vars=scenario_names, var_name='Scenario', value_name='Rate')

# build charts ---

plot_left = alt.Chart(df_left_long).mark_line().encode(
    x=alt.X('Date:T', title='Maturity'),
    y=alt.Y('Rate:Q',
            title='Rate / Yield (Decimal)',
            axis=alt.Axis(format='%'),
            scale=alt.Scale(zero=False)
           ),
    color=alt.Color('Scenario:N', title='Scenario'),
    tooltip=['Scenario:N', 'Date:T', 'Rate:Q']
).properties(
    title='Spot Rates',
    width=300,
    height=430
)

plot_right = alt.Chart(df_right_long).mark_line().encode(
    x=alt.X('Date:T', title='Maturity'),
    y=alt.Y('Rate:Q', 
            title=None, # Hidden Y-axis title
            axis=alt.Axis(format='%') # ADDED: Matches the left plot's percentage format
           ), 
    color=alt.Color('Scenario:N', title='Scenario'),
    tooltip=['Scenario:N', 'Date:T', 'Rate:Q']
).properties(
    title=' Yield to Maturity',
    width=300,
    height=430
)

# combine and lock the Y-axis
final_chart = (plot_left | plot_right).resolve_scale(y='shared')
final_chart

alt.HConcatChart(...)

## Conclusion: Comparing Spot Rates to Yield to Maturity 📍 ⚖️ 🎯
The side-by-side plots visually confirm the mathematical relationship between the underlying term structure (Spot Rates) and the market's summarized pricing metric (Yield to Maturity).

By running our `optimize.newton` solver across six distinct interest rate regimes, two critical observations emerge:

### 1. 🔗 YTM Inherits the Term Structure's Shape
In every scenario, the Yield to Maturity curve adopts the general posture of its parent term structure. An upward-sloping spot curve (like the Base Case or Bear Steepener) generates an upward-sloping YTM curve. Conversely, an inverted spot curve forces the YTM curve to invert as well.

### 2. ⏳ The "Lag" Effect (YTM as an Average)
While the shapes match, the YTM curve is distinctly muted compared to the spot curve. It "lags" behind the extreme ends of the term structure.

* **Upward-Sloping Scenarios**: Look at the Bear Steepener (red line). At the 2055 maturity, the spot rate hits roughly 7.0%, but the YTM for that exact same maturity maxes out around 6.7%. The YTM is lower.

* **In Inverted Scenarios**: Look at the Inverted curve (green line). At the 2055 maturity, the spot rate drops to approximately 3.3%, but the YTM settles higher, near 3.4%.

#### 🤔💭 Why does this happen?
This lag perfectly illustrates the core limitation of Yield to Maturity discussed in the introduction. YTM acts as a single, uniform average discount rate applied to all of a bond's cash flows.

When you price a 30-year bond on a steep upward-sloping curve, the principal repayment in year thirty is heavily discounted by the high thirty-year spot rate. The coupons, they are paid in years one, two, and three and discounted at much lower short-term spot rates. Because the YTM balances the present value of all the cash flows against the market price, the low rates applied to the early coupons drag the average YTM down below the peak value of the thirty-year spot rate.